# 03 Bowl source: a focused spherical bowl

| | |
|---|---|
| **Purpose** | The standard single-element focused ultrasound transducer: a spherical section bowl. Shows focusing, the on-axis gain, and where `location` puts the bowl. |
| **Inputs** | 1 kHz bowl, aperture (outer) radius 0.8 m, radius of curvature 1 m, apex at (-4, 0, 0), axis +x, normal velocity 1 m/s; bone sphere of radius 1 m at the origin, element size 0.4 m, in water. |
| **Outputs** | On-axis comparison with O'Neil's formula; incident and total pressure on z = 0 (`outputs/03_bowl_source_*.png`). |
| **Checks** | On-axis pressure converges to O'Neil's analytical solution as the source is refined; the on-axis maximum lies between the apex and the geometric focus; GMRES converged; field finite. |
| **Runtime** | About 25 s full, 15 s CI mode (Docker image, 4 CPUs, 9 Nov 2026). CI mode uses a coarser plotting grid. |
| **Adapted from** | `notebooks/Bowl source.ipynb` (upstream OptimUS). The upstream notebook uses outer radius = radius of curvature = 1 m (a full hemisphere); here the aperture is 0.8 m so the bowl is a typical cap, not a hemisphere. |

## Physics notes

For a spherical cap with radius of curvature $d$, aperture radius $a$, depth $h = d - \sqrt{d^2 - a^2}$
and uniform normal velocity $U_0$, O'Neil (1949, *J. Acoust. Soc. Am.* 21, 516) derived the on-axis pressure

$$ |p(z)| = \left| \frac{2 \rho_0 c U_0}{1 - z/d} \sin\left( \tfrac{k}{2} (r_e - z) \right) \right|,
   \qquad r_e = \sqrt{(z - h)^2 + a^2} $$

with $z$ measured from the apex. At the geometric focus $z = d$ this becomes $|p| = \rho_0 c U_0 k h$,
the focal gain. When $kh$ is small (a weakly focused bowl, as here at 1 kHz) the on-axis maximum sits
*before* the geometric focus: the focal shift described by O'Neil and by Lucas and Muir (1982,
*J. Acoust. Soc. Am.* 72, 1289).

**Note on `location`.** The `create_bowl` docstring says `location` is the centroid of the bowl,
but the code places the **apex** there. The agreement with O'Neil's formula below, which measures
$z$ from the apex, confirms this.

> **To expand (Yash):** derive the focal gain $\rho_0 c U_0 k h$; compute the focal shift for this bowl and compare with the measured on-axis maximum; discuss what happens at clinical frequencies (about 1 MHz).

## Setup

Imports, the `CI_MODE` switch and the output folder.

In [ ]:
NOTEBOOK_NAME = "03_bowl_source"

%matplotlib inline
import json
import os
import time

import numpy as np
import matplotlib.pyplot as plt

import optimus

# CI_MODE uses smaller settings so the notebook finishes in a few minutes in CI.
# Set it from the shell with OPTIMUS_CI_MODE=1, or change it here by hand.
CI_MODE = os.environ.get("OPTIMUS_CI_MODE", "0") == "1"

# Figures and a small JSON file of key results are written here.
OUTPUT_DIR = os.environ.get("OPTIMUS_OUTPUT_DIR", "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

RESULTS = {"notebook": NOTEBOOK_NAME, "ci_mode": CI_MODE,
           "optimus_version": optimus.__version__, "checks": {}}
START_TIME = time.time()

print("OptimUS", optimus.__version__, "| CI mode:", CI_MODE, "| outputs in:", OUTPUT_DIR)

In [ ]:
def check(name, condition, detail):
    """Record a named check and stop the notebook if it fails."""
    RESULTS["checks"][name] = bool(condition)
    print(("PASS  " if condition else "FAIL  ") + name + ": " + detail)
    assert condition, name + " failed: " + detail


def gmres_converged(model):
    """GMRES stops early (before maxiter) only when the tolerance is reached."""
    if hasattr(model, "linear_solve_iteration_count"):
        iterations = model.linear_solve_iteration_count
    else:
        iterations = model.iteration_count
    return iterations, iterations < optimus.global_parameters.linalg.maxiter


def save_figures(figs, stem):
    """Save a list of matplotlib figures as PNG files in OUTPUT_DIR."""
    if not isinstance(figs, (list, tuple)):
        figs = [figs]
    for i, fig in enumerate(figs):
        fig.savefig(os.path.join(OUTPUT_DIR, "%s_%d.png" % (stem, i + 1)), dpi=110,
                    bbox_inches="tight")

## 1. Parameters

In [ ]:
frequency = 1000.0                 # Hz
outer_radius = 0.8                 # m, aperture radius a
radius_of_curvature = 1.0          # m, d
bowl_location = (-4.0, 0.0, 0.0)   # apex of the bowl (see the note above)
velocity = 1.0                     # m/s
element_size = 0.4                 # m, mesh of the bone sphere
grid_resolution = (100, 75) if CI_MODE else (240, 180)
bounding_box = (-4, 4, -3, 3)

## 2. Source, geometry and materials

In [ ]:
source = optimus.source.create_bowl(frequency, outer_radius=outer_radius,
                                    radius_of_curvature=radius_of_curvature,
                                    location=bowl_location, velocity=velocity, inner_radius=0)
geometry = optimus.geometry.shapes.Sphere(element_size=element_size)
material_ext = optimus.material.load_material("water")
material_int = optimus.material.load_material("bone (cortical)")

## 3. On-axis pressure against O'Neil's formula

In [ ]:
rho, c = material_ext.density, material_ext.speed_of_sound
k = 2 * np.pi * frequency / c
d, a = radius_of_curvature, outer_radius
h = d - np.sqrt(d**2 - a**2)

z = np.linspace(0.05, 5.0, 200)
z = z[np.abs(z - d) > 1e-3]                    # the formula is 0/0 exactly at the focus
axis_points = np.vstack((bowl_location[0] + z, 0 * z, 0 * z))
r_e = np.sqrt((z - h)**2 + a**2)
p_oneil = np.abs(2 * rho * c * velocity / (1 - z / d) * np.sin(k / 2 * (r_e - z)))
focal_gain_analytical = rho * c * velocity * k * h

def on_axis(points_per_wavelength):
    s = optimus.source.create_bowl(frequency, outer_radius=outer_radius,
                                   radius_of_curvature=radius_of_curvature,
                                   location=bowl_location, velocity=velocity, inner_radius=0,
                                   number_of_point_sources_per_wavelength=points_per_wavelength)
    p = np.abs(s.pressure_field(material_ext, axis_points))
    return p, np.max(np.abs(p - p_oneil)) / np.max(p_oneil)

p_default, error_default = on_axis(6)
p_fine, error_fine = on_axis(24)
z_peak = z[np.argmax(p_fine)]
RESULTS["on_axis_error_6ppw"] = round(float(error_default), 4)
RESULTS["on_axis_error_24ppw"] = round(float(error_fine), 4)
RESULTS["on_axis_peak_distance_from_apex_m"] = round(float(z_peak), 3)
print("Bowl depth h = %.3f m, analytical focal pressure = %.2f MPa" % (h, focal_gain_analytical / 1e6))
print("Max on-axis error: %.3f (6 per wavelength), %.3f (24 per wavelength)" % (error_default, error_fine))
print("On-axis maximum at %.2f m from the apex (geometric focus at %.2f m)" % (z_peak, d))

fig = plt.figure(figsize=(7, 4))
plt.plot(z, p_oneil / 1e6, "k-", label="O'Neil (1949)")
plt.plot(z, p_default / 1e6, "o", ms=3, label="OptimUS, 6 per wavelength")
plt.plot(z, p_fine / 1e6, "x", ms=3, label="OptimUS, 24 per wavelength")
plt.axvline(d, color="grey", ls="--", lw=1, label="geometric focus")
plt.xlabel("distance from apex z [m]")
plt.ylabel("|p| [MPa]")
plt.legend()
plt.grid(alpha=0.3)
save_figures(fig, NOTEBOOK_NAME + "_axis")
plt.show()

## 4. Model and solve

In [ ]:
model = optimus.model.create_default_model(source, geometry, material_ext, material_int)
t0 = time.time()
model.solve()
RESULTS["solve_time_s"] = round(time.time() - t0, 2)
iterations, converged = gmres_converged(model)
RESULTS["gmres_iterations"] = int(iterations)
print("GMRES iterations:", iterations)

## 5. Field on the plane z = 0

In [ ]:
plane = optimus.postprocess.VisualisePlane(model)
plane.create_computational_grid(resolution=grid_resolution, bounding_box=bounding_box)
plane.compute_fields()
figs = optimus.postprocess.plot_pressure_field(plane, field="incident", unit="MPa")
save_figures(figs, NOTEBOOK_NAME + "_incident")
figs = optimus.postprocess.plot_pressure_field(plane, field="total", unit="MPa")
save_figures(figs, NOTEBOOK_NAME + "_total")

## 6. Checks

Measured on 9 Nov 2026 in the Docker image for this bowl: on-axis error 8.3% at 6 point sources per
wavelength and 0.5% at 24. Limits: 12% and 2%, and the finer source must be more accurate.

In [ ]:
check("source_default_accuracy", error_default < 0.12,
      "on-axis error %.4f at 6 per wavelength, limit 0.12" % error_default)
check("source_converges_to_oneil", error_fine < 0.02 and error_fine < error_default,
      "on-axis error %.4f at 24 per wavelength, limit 0.02" % error_fine)
check("prefocal_peak", 0 < z_peak < d,
      "on-axis maximum at %.2f m, between apex (0) and focus (%.2f m)" % (z_peak, d))
check("gmres_converged", converged, "%d iterations" % iterations)
total = plane.total_field_imshow
check("field_is_finite", np.all(np.isfinite(total)),
      "%d NaN or inf values on the plane" % np.sum(~np.isfinite(total)))

## Summary

Key numbers and check results, also saved as JSON in `OUTPUT_DIR` so CI and the assistant can read them.

In [ ]:
RESULTS["runtime_s"] = round(time.time() - START_TIME, 1)
with open(os.path.join(OUTPUT_DIR, NOTEBOOK_NAME + "_results.json"), "w") as f:
    json.dump(RESULTS, f, indent=2, sort_keys=True)
print(json.dumps(RESULTS, indent=2, sort_keys=True))